# datalake NYC Taxi × Météo

## Contexte

La NYC Taxi & Limousine Commission (TLC) publie chaque mois, depuis 2009, l'intégralité des courses de taxis et de véhicules de transport avec chauffeur effectuées à New York. Ce jeu de données est massif et a etait modifier plusieur fois a travers les années 


---
## Les quatre types de véhicules

| Type | Début de publication | Remarque |
|---|---|---|
| **Yellow taxi** | Janvier 2009 | Le plus ancien, le plus documenté |
| **Green taxi** | Août 2013 | Créé pour desservir les zones hors Manhattan central |
| **FHV** (For-Hire Vehicle) | 2015 | Véhicules privé, ensuite uniquement Véhicules de tourisme/limousines |
| **FHVHV** (High Volume For-Hire Vehicle) | Février 2019 | Créé par la loi locale 149 (2018) pour les plateformes dispatchant plus de 10 000 courses/jour (Uber et Lyft) |

**Point d'attention :** depuis 2019 FHV capte seulement les courses des plateformes qui dépassent le seuil des 10 000 courses/jour, tandis que FHV continue de couvrir le reste des bases de VTC/limousines. FHV est structurellement beaucoup plus pauvre : il ne contient ni tarif, ni distance (sub based service)


## Localisation des prises en charge / dépose

Jusqu'en juin 2016, les fichiers contiennent des **coordonnées GPS brutes** (latitude/longitude) pour la prise en charge et la dépose. À partir de juillet 2016, la TLC ne publie plus de coordonnées GPS : chaque course référence à la place un **identifiant de zone** (`LocationID`), qui renvoie vers une table de référence publiée séparément par la TLC (zones, arrondissements, géométries).

## Colonnes apparues au fil du temps

Plusieurs colonnes tarifaires sont apparues progressivement dans le schéma (elles n'existaient tout simplement pas dans les fichiers plus anciens) :

- des frais aéroport
- un supplément d'amélioration du service, introduit en 2015 ;
- un supplément de congestion (congestion pricing), introduit en 2019 ;
- un nouveau supplément lié à la zone de tarification de congestion de Manhattan (« CBD »), en vigueur depuis le 5 janvier 2025.

## Fréquence de publication

Les données sont publiées mensuellement, avec un délai d'environ deux mois (les données de janvier sortent typiquement fin février/début mars).

---
# Bronze : ingestion et organisation dans HDFS

1. **Concevez une convention de nommage des chemins** dans HDFS pour organiser les données brutes. On ne vous dit pas quelle convention utiliser — mais votre convention doit permettre de répondre facilement, sans avoir à lire le contenu des fichiers, aux questions suivantes :
   - Quels types de véhicules ai-je déjà ingérés ?
   - Pour quelle période (année/mois) ai-je des données pour un type donné ?
   - Est-ce qu'un mois précis d'un type précis a déjà été ingéré (pour ne pas le re-télécharger) ?

2. **Écrivez le script d'ingestion** qui télécharge les fichiers et les dépose dans HDFS. Points critique :
   - il doit pouvoir être interrompu et relancé sans tout retélécharger ;
   - il doit gérer le fait que chaque type de véhicule n'existe pas depuis la même date;

3. **Ingérez aussi la météo.** source de données météo historiques horaires pour New York (Open-Meteo ou de la NOAA) le type de persitence pour les données meteroloique devrait etre logique par rapport a la source utilisé (API / parquet).


In [ ]:
# ingestion des données 

In [ ]:
# ingestion de la météo vers HDFS.

---
# Silver : nettoyage, réconciliation et modélisation


## Choisissez votre système de stockage



### Réconcilier les deux systèmes de localisation

Rappel : coordonnées GPS brutes avant juillet 2016, identifiants de zone après. Vous devez faire en sorte que toutes vos courses, quelle que soit leur date, puissent être rattachées à une même notion de zone/arrondissement.


### Gérer les colonnes apparues progressivement

Rappel : plusieurs suppléments tarifaires n'existaient pas dans les fichiers les plus anciens. Une valeur manquante *avant* l'introduction d'un supplément et une valeur de 0 *après* son introduction ne signifient pas la même chose. Assurez-vous que votre pipeline ne confond pas les deux pour bien les prendre en consideration plus tard au niveau des insights



votre modèle silver doit permettre de répondre, sans retourner aux fichiers bronze, à des questions comme : 

*combien de courses par type de véhicule et par mois*, 
*quelle est la zone de prise en charge la plus fréquente*, 
*quelle est la météo au moment d'une course donnée*. 

### modélisation silver


*(à compléter)*

In [ ]:
# Exercice : job Spark qui lit bronze, applique vos règles de nettoyage/réconciliation, 
# et écrit dans le système que vous avez choisi pour les véhicules et la meteo.
# À COMPLÉTER

---
# Gold : des tables construites pour des questions précises

partez de la question, et construisez la table (ou la requête) qui y répond directement.

Voici les quatre analyses que votre datalake doit permettre.

## 1. Évolution des nouveaux suppléments tarifaires

**Objectif :** pour chaque supplément apparu progressivement dans le schéma, montrer son évolution dans le temps (montant total ou moyen collecté), par type de véhicule.

## 2. Flux de prises en charge / déposes (diagramme en corde)

**Objectif :** visualiser, sous forme de chord diagram, les flux de courses entre zones de départ et d'arrivée.

## 3. Évolution du prix des courses par zone de départ (ridgeline plot)

**Objectif :** un ridgeline plot montrant l'evolution des prix par km au fil des années, pour une zone de départ donnée.

## 4. Fréquence des courses par heure et jour de la semaine (heatmap)

**Objectif :** une fonction qui, pour une année donnée en paramètre, produit une heatmap (jour de la semaine × heure de la journée) de la fréquence des courses, pour chaque type de véhicule.

## 5. Écart de prix selon la météo

**Objectif :** comparer le prix des courses (par km) selon les conditions météo au moment de la course, par rapport au prix moyen général 

## À vous de jouer

Pour chacune des quatre analyses : concevez la ou les tables gold nécessaires (ou la requête directe sur silver, si vous jugez qu'une table gold dédiée n'est pas nécessaire !!!!!!) 

In [ ]:
# Exercice : Évolution des nouveaux suppléments tarifaires
# À COMPLÉTER

In [ ]:
# Exercice : Diagramme en corde des flux pickup/dropoff
# À COMPLÉTER

In [ ]:
# Exercice : Ridgeline plot de l'évolution du prix par zone de départ
# À COMPLÉTER

In [ ]:
# Exercice : Fonction heatmap(annee) : fréquence des courses par jour/heure et par type de véhicule
# À COMPLÉTER
